## Create client

In [1]:
from scray.job_client.config import ScrayJobClientConfig
from scray.job_client.client import ScrayJobClient 

config = ScrayJobClientConfig(
  host_address = "http://ml-integration.research.dev.seeburger.de",
  #host_address = "http://localhost",
  port = 8082
)

client = ScrayJobClient(config=config)

In [ ]:
env = "http://scray.org/ai/jobs/env/see/ticket-project/provence-data/prod/ticket_update"
 
# Blocking till new job apears
#jobs = client.get_jobs(processing_env=env, requested_state="FINISHED") 

#print(len(jobs))
#for job_name in jobs:#    print("Process uploaded data from job: " + job_name)

In [2]:
env="http://scray.org/ai/jobs/env/see/ticket-project/provence-data/prod/last_download"
job_name="last_download"
print(client.get_job_state(job_name=job_name))

http://ml-integration.research.dev.seeburger.de:8082/sync/versioneddata/latest?datasource=last_download&mergekey=_
1757484495.0


In [ ]:
client.get_job_metadata(job_name='ticket_update-3008af1c-8e19-4dec-98f3-e01e3b248855')

In [ ]:
import time

jobs1_times = []
jobs2_times = []

for i in range(1, 11):
    print(f"\nRun {i}:")

    start = time.time()
    jobs1 = client.get_jobs(processing_env=env)
    end = time.time()
    t1 = end - start
    jobs1_times.append(t1)
    print(f"Fetching jobs1 consumed time = {t1:.4f} seconds, count = {len(jobs1)}")

    start = time.time()
    jobs2 = client.get_jobs(processing_env=env, requested_state="FINISHED")
    end = time.time()
    t2 = end - start
    jobs2_times.append(t2)
    print(f"Fetching jobs2 consumed time = {t2:.4f} seconds, count = {len(jobs2)}")

# --- Summary statistics ---
def summarize(times, label):
    avg = sum(times) / len(times)
    print(
        f"\n{label} statistics over {len(times)} runs:\n"
        f"  Average: {avg:.4f} seconds\n"
        f"  Min:     {min(times):.4f} seconds\n"
        f"  Max:     {max(times):.4f} seconds"
    )

summarize(jobs1_times, "Client filterd fetch")
summarize(jobs2_times, "Indexed state fetch")


In [ ]:
from collections import Counter

def countStates(state):
    jobs_new = client.get_jobs(processing_env=env, requested_state=state)
    jobs_old = client.get_jobs_old(processing_env=env, requested_state=state)

    #for job in jobs_new:
    #    client.get_job_metadata(job_name=job)

    print("---------------------------------------------------------------------------")
    print(f"State: {state}")
    print(f"\tNew (get_jobs): {len(jobs_new)}")
    print(f"\tOld (get_jobs_old): {len(jobs_old)}")

    # Differences
    only_in_new = set(jobs_new) - set(jobs_old)
    only_in_old = set(jobs_old) - set(jobs_new)

    if only_in_new:
        print("\nJobs only in get_jobs:")
        for job in sorted(only_in_new):
            print(f"  - {job}")

    if only_in_old:
        print("\nJobs only in get_jobs_old:")
        for job in sorted(only_in_old):
            print(f"  - {job}")

    if not only_in_new and not only_in_old:
        print("\n✅ Both lists contain the same jobs.")

    print("---------------------------------------------------------------------------")

    # return both lists if needed
    return jobs_new, jobs_old


In [ ]:
countStates("CERTIFICATE_TO_MFT/INSTALL")
countStates("UPDATED")
countStates("FINISHED")
countStates("PUBLISHED")

In [3]:
import time


jobs2_times = []

env = "http://scray.org/ai/jobs/env/see/ticket-project/provence-data/prod/ticket_update"

def print_num_states(state):
    jobs2 = client.get_jobs(processing_env=env, requested_state=state)
    print("State: " + state)
    print("\tNum jobs " + str(len(jobs2)))
    #print("\tNames " + str(jobs2))

#print_num_states("CERTIFICATE_TO_MFT/INSTALL")
print_num_states("UPDATED")
#print_num_states("FINISHED")
#print_num_states("FINISHED")

State: UPDATED
	Num jobs 25911


In [4]:
import time


jobs2_times = []

env = "http://scray.org/ai/jobs/env/see/ticket-project/provence-data/prod/ticket_update"

def cancelJobs(state):
    jobs = client.get_jobs(processing_env=env, requested_state=state)
    print("State: " + state)
    print("\tNum jobs " + str(len(jobs)))
    total_jobs = len(jobs)

    for idx in range(total_jobs):
        job_name = jobs[idx]
        if (idx + 1) % 100 == 0 or (idx + 1) == total_jobs:
            print(f"\t[{idx + 1}/{total_jobs}]", flush=True)
        # Set a new state of the job
        client.setState(job_name=job_name, processing_env=env, state="CANCELED")

#print_num_states("CERTIFICATE_TO_MFT/INSTALL")
print_num_states("UPDATED")
cancelJobs("UPDATED")
print_num_states("UPDATED")
#print_num_states("FINISHED")


State: UPDATED
	Num jobs 26053
State: UPDATED
	Num jobs 26053
	[100/26053]
	[200/26053]
	[300/26053]


KeyboardInterrupt: 